# Quality Metrics Showcase — All Three Metrics

This notebook demonstrates the **three quality metrics** in mycontext-ai:

1. **QualityMetrics** — Score context/prompt quality across 6 dimensions *before* execution
2. **OutputEvaluator** — Score LLM output quality across 5 dimensions *after* execution
3. **Context Amplification Index (CAI)** — Prove templates improve output (templated vs raw)

**Setup:** `pip install mycontext-ai` (or `pip install -e .` from repo root).  
**API:** Heuristic mode needs no API key. LLM mode and CAI require `OPENAI_API_KEY`.

In [1]:
from mycontext import Context
from mycontext.intelligence import (
    QualityMetrics,
    QualityDimension,
    OutputEvaluator,
    OutputDimension,
    ContextAmplificationIndex,
    CAIResult,
)
from mycontext.templates.free.reasoning import RootCauseAnalyzer

print("Imports OK")

Imports OK


---
## 1. QualityMetrics — Score Context Before Execution

**Six dimensions:** Clarity, Completeness, Specificity, Relevance, Structure, Efficiency  
**Three modes:** `heuristic` (free, instant), `llm` (accurate, ~$0.02/eval), `hybrid` (best of both)

In [2]:
# Build a context with Root Cause Analyzer
ctx = RootCauseAnalyzer().build_context(
    problem="API response times tripled after last deployment",
    depth="thorough",
)

metrics = QualityMetrics(mode="heuristic")
score = metrics.evaluate(ctx)

print(f"Overall: {score.overall:.1%}")
print()
print(metrics.report(score))

Overall: 87.2%

Context Quality Report

Overall Score: 87.2% ✅

Dimension Scores:
  ✅ Clarity: 100.0%
  ✅ Completeness: 85.0%
  ✅ Specificity: 90.0%
  ⚠️ Relevance: 70.0%
  ✅ Structure: 100.0%
  ⚠️ Efficiency: 70.0%

Strengths (25):
  ✓ No vague language
  ✓ Clear role and directive structure
  ✓ Output format specified
  ✓ Low pronoun ratio — clear, unambiguous references
  ✓ Low hedging — instructions are binding and clear
  ✓ Sufficient detail for clear instructions
  ✓ Specific role/guidance defined
  ✓ Substantive directive
  ✓ Clear goal/task defined
  ✓ Well-defined rules (6)
  ✓ Constraints defined
  ✓ Uses specific, concrete language
  ✓ Rich domain-specific terminology
  ✓ Detailed, comprehensive directive
  ✓ Well-defined constraints add specificity
  ✓ Behavioral rules add specificity
  ✓ Includes measurable criteria
  ✓ Comprehensive directive with organized sections
  ✓ Constraints help maintain focus
  ✓ Role-directive alignment
  ✓ Well-organized with clear sections
  ✓

In [3]:
# Compare two contexts — see improvement
raw = Context(guidance="Expert", directive="Analyze API slowdown")
templated = RootCauseAnalyzer().build_context(
    problem="API response times tripled after deployment",
    depth="thorough",
)

diff = metrics.compare(raw, templated)
print(f"Original: {diff['original_score']:.1%}")
print(f"Improved: {diff['improved_score']:.1%}")
print(f"Improvement: {diff['improvement_percentage']:+.1f}pp")
print("\nDimension changes:")
for dim, delta in diff["dimension_changes"].items():
    print(f"  {dim.value}: {delta:+.2f}")

Original: 0.0%
Improved: 87.2%
Improvement: +87.2pp

Dimension changes:
  clarity: +1.00
  completeness: +0.85
  specificity: +0.90
  relevance: +0.70
  structure: +1.00
  efficiency: +0.60


---
## 2. OutputEvaluator — Score LLM Output After Execution

**Five dimensions:** Instruction Following, Reasoning Depth, Actionability, Structure Compliance, Cognitive Scaffolding  
Evaluates how well the LLM response leveraged the context.

In [4]:
# Build context and simulate output (or use ctx.execute() with API key)
ctx = RootCauseAnalyzer().build_context(
    problem="API response times tripled after deployment",
    depth="thorough",
)

# Example output (in practice, use ctx.execute(provider="openai"))
sample_output = """
## Root Cause Analysis

1. **Immediate causes:** Database connection pool exhaustion (40% increase in connections).
2. **Contributing factors:** New feature deployed without load testing; connection timeout too low.
3. **Recommendations:** Increase pool size, add connection retry logic, run load tests before deploy.
"""

evaluator = OutputEvaluator(mode="heuristic")
score = evaluator.evaluate(ctx, sample_output)

print(f"Output quality: {score.overall:.1%}")
print()
print(evaluator.report(score))

Output quality: 39.7%

Output Quality Report

Overall: 39.7%

Dimensions:
  [-] Instruction Following: 15.0%  (Matched 0/2 action verbs, 0/1 required terms | Instruction coverage: 6/16 items)
  [~] Reasoning Depth: 41.6%  (2 reasoning markers, 3 numbered steps, 1 section headings)
  [-] Actionability: 35.0%  (2 action phrases, 1 action items, 1 concrete metrics)
  [+] Structure Compliance: 75.0%  (Well-structured output)
  [~] Cognitive Scaffolding: 46.7%  (Output uses 3/9 cognitive frameworks from context)

Strengths:
  + Strong Structure Compliance

Weaknesses:
  - Weak Instruction Following
  - Weak Actionability


In [5]:
# Evidence per dimension
for dim in OutputDimension:
    ev = score.evidence.get(dim, "")
    val = score.dimensions.get(dim, 0)
    print(f"{dim.value}: {val:.1%} — {ev}")

instruction_following: 15.0% — Matched 0/2 action verbs, 0/1 required terms | Instruction coverage: 6/16 items
reasoning_depth: 41.6% — 2 reasoning markers, 3 numbered steps, 1 section headings
actionability: 35.0% — 2 action phrases, 1 action items, 1 concrete metrics
structure_compliance: 75.0% — Well-structured output
cognitive_scaffolding: 46.7% — Output uses 3/9 cognitive frameworks from context


---
## 3. Context Amplification Index (CAI) — Prove Templates Work

**CAI = templated_score / raw_score**  
- CAI > 1.0 → template improved output  
- CAI = 1.0 → neutral  
- CAI < 1.0 → template hurt output  

**Requires API key** — runs the question twice (raw + templated), scores both.

In [6]:
# CAI measurement (requires OPENAI_API_KEY)
import os

if os.environ.get("OPENAI_API_KEY"):
    cai = ContextAmplificationIndex(provider="openai", eval_mode="heuristic")
    result = cai.measure(
        question="Why did API response times triple after the last deployment?",
        template_name="root_cause_analyzer",
    )
    print(f"CAI: {result.cai_overall:.2f}x  ({result.verdict})")
    print()
    print(cai.report(result))
else:
    print("Set OPENAI_API_KEY to run CAI measurement.")
    print("CAI runs the question raw + templated, scores both, returns the ratio.")

Set OPENAI_API_KEY to run CAI measurement.
CAI runs the question raw + templated, scores both, returns the ratio.


In [ ]:
# CAI for chains — compare single template vs multi-template chain
if os.environ.get("OPENAI_API_KEY"):
    result = cai.measure_chain(
        question="Why did customer churn spike 40% last quarter?",
        chain=["root_cause_analyzer", "decision_framework"],
    )
    print(f"Chain vs single: {result.cai_overall:.2f}x  ({result.verdict})")
    print(f"Metadata: {result.metadata}")
else:
    print("Set OPENAI_API_KEY to run measure_chain().")

---
## 4. Combined Workflow — Measure Before, Execute, Measure After

1. **QualityMetrics** — score context before sending  
2. Execute with your LLM  
3. **OutputEvaluator** — score the output  
4. **CAI** — compare raw vs templated (optional)

In [7]:
# Full pipeline (heuristic only — no API needed)
ctx = RootCauseAnalyzer().build_context(
    problem="Database queries are 5x slower after schema migration",
    depth="thorough",
)

# 1. Score context before execution
qm = QualityMetrics(mode="heuristic")
ctx_score = qm.evaluate(ctx)
print("=== Before Execution ===")
print(f"Context quality: {ctx_score.overall:.1%}")
if ctx_score.issues:
    print("Issues:", ctx_score.issues[:3])
print()

# 2. Simulate output (replace with ctx.execute(provider="openai") for real run)
mock_output = """Root causes: 1) Missing indexes on new columns. 2) N+1 queries in ORM. 3) Connection pool too small.
Recommendations: Add indexes, use eager loading, increase pool size to 50."""

# 3. Score output after execution
oe = OutputEvaluator(mode="heuristic")
out_score = oe.evaluate(ctx, mock_output)
print("=== After Execution ===")
print(f"Output quality: {out_score.overall:.1%}")
print("Strengths:", out_score.strengths)
print("Weaknesses:", out_score.weaknesses)

=== Before Execution ===
Context quality: 87.2%
Issues: ['No examples -- add concrete examples of expected input/output', 'No examples or concrete specifics', 'Directive is long (418 words) — shorter, tighter directives correlate with better LLM accuracy; move detail into rules/constraints']

=== After Execution ===
Output quality: 24.9%
Strengths: []
Weaknesses: ['Weak Instruction Following', 'Weak Reasoning Depth', 'Weak Actionability', 'Weak Cognitive Scaffolding']


---
## 5. Mode Comparison — Heuristic vs LLM vs Hybrid

| Mode | Speed | Cost | Best for |
|------|-------|------|----------|
| `heuristic` | Instant | Free | CI/CD, bulk eval, dev |
| `llm` | ~2s | ~$0.02/eval | Production QA, authoritative scoring |
| `hybrid` | Fast | Low | Borderline cases (0.45–0.75) get LLM |

In [8]:
# All three metrics support mode="heuristic" | "llm" | "hybrid"
ctx = RootCauseAnalyzer().build_context(problem="Service outages increased", depth="thorough")

# QualityMetrics
qm_h = QualityMetrics(mode="heuristic")
qm_llm = QualityMetrics(mode="llm")  # needs API key

# OutputEvaluator
oe_h = OutputEvaluator(mode="heuristic")
oe_llm = OutputEvaluator(mode="llm")  # needs API key

# CAI uses eval_mode for how to score outputs
# cai = ContextAmplificationIndex(provider="openai", eval_mode="heuristic")

s = qm_h.evaluate(ctx)
print(f"QualityMetrics (heuristic): {s.overall:.1%}")
print(f"Dimensions: {list(s.dimensions.keys())}")

QualityMetrics (heuristic): 87.2%
Dimensions: [<QualityDimension.CLARITY: 'clarity'>, <QualityDimension.COMPLETENESS: 'completeness'>, <QualityDimension.SPECIFICITY: 'specificity'>, <QualityDimension.RELEVANCE: 'relevance'>, <QualityDimension.STRUCTURE: 'structure'>, <QualityDimension.EFFICIENCY: 'efficiency'>]


---
## Summary

- **QualityMetrics** — 6 dimensions, score context *before* you send a token  
- **OutputEvaluator** — 5 dimensions, score LLM response *after* execution  
- **CAI** — Prove templates work: run raw vs templated, compare scores  

All three support `heuristic` (free), `llm` (accurate), and `hybrid` (best of both).